In [1]:
from pathlib import Path
import pandas as pd

raw_folder = Path("../Data/raw")
processed_folder = Path("../Data/processed")

In [2]:
airfare = pd.read_csv(
    processed_folder / "top10_airfare_2022_2025.csv"
)

airfare

,Dest,FARE_2022,FARE_2023,FARE_2024,FARE_2025
0,LAS,147.90,139.33,144.77,170.65
1,SFO,154.75,169.95,177.04,192.84
2,DEN,163.52,161.81,163.03,173.53
3,PDX,178.91,166.86,152.47,191.22
4,SAN,182.47,171.90,175.79,221.54
5,PHX,191.28,158.87,164.62,185.09
6,LAX,205.28,149.98,169.38,206.81
7,ANC,219.94,205.09,263.05,282.22
8,ORD,223.18,201.64,217.43,233.31
9,DFW,263.67,242.63,207.70,227.33


In [3]:
t100_files = sorted(
    raw_folder.glob("t100_dec_*.zip")
)

t100_all = pd.concat(
    [
        pd.read_csv(file, compression="zip")
        for file in t100_files
    ],
    ignore_index=True
)

sea_all = t100_all[
    t100_all["ORIGIN"] == "SEA"
].copy()

sea_all.shape

(2431, 19)

In [4]:
top10_airports = [
    "LAX", "PHX", "LAS", "ANC", "DEN",
    "SFO", "DFW", "ORD", "SAN", "PDX"
]

sea_top10 = sea_all[
    sea_all["DEST"].isin(top10_airports)
].copy()

sea_top10.shape

(673, 19)

In [5]:
destination_metrics = (
    sea_top10
    .groupby(
        ["DEST", "DEST_CITY_NAME", "DEST_STATE_ABR"],
        as_index=False
    )
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum"),
        DISTANCE=("DISTANCE", "mean")
    )
)

destination_metrics["LOAD_FACTOR_PCT"] = (
    destination_metrics["PASSENGERS"]
    / destination_metrics["SEATS"]
    * 100
).round(1)

destination_metrics

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS,FLIGHTS,SEATS,DISTANCE,LOAD_FACTOR_PCT
0,ANC,"Anchorage, AK",AK,278351.0,2362.0,331528.0,1448.0,84.0
1,DEN,"Denver, CO",CO,269743.0,2001.0,312259.0,1024.0,86.4
2,DFW,"Dallas/Fort Worth, TX",TX,237870.0,1497.0,268959.0,1660.0,88.4
3,LAS,"Las Vegas, NV",NV,299459.0,2051.0,350437.0,867.0,85.5
4,LAX,"Los Angeles, CA",CA,315904.0,2493.0,373393.0,954.0,84.6
5,ORD,"Chicago, IL",IL,205165.0,1442.0,239114.0,1721.0,85.8
6,PDX,"Portland, OR",OR,178619.0,2401.0,238323.0,129.0,74.9
7,PHX,"Phoenix, AZ",AZ,308110.0,2375.0,372050.0,1107.0,82.8
8,SAN,"San Diego, CA",CA,194524.0,1443.0,233094.0,1050.0,83.5
9,SFO,"San Francisco, CA",CA,261444.0,2181.0,321273.0,679.0,81.4


In [6]:
master_top10 = destination_metrics.merge(
    airfare,
    left_on="DEST",
    right_on="Dest",
    how="left"
)

master_top10 = master_top10.drop(
    columns="Dest"
)

master_top10

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS,FLIGHTS,SEATS,DISTANCE,LOAD_FACTOR_PCT,FARE_2022,FARE_2023,FARE_2024,FARE_2025
0,ANC,"Anchorage, AK",AK,278351.0,2362.0,331528.0,1448.0,84.0,219.94,205.09,263.05,282.22
1,DEN,"Denver, CO",CO,269743.0,2001.0,312259.0,1024.0,86.4,163.52,161.81,163.03,173.53
2,DFW,"Dallas/Fort Worth, TX",TX,237870.0,1497.0,268959.0,1660.0,88.4,263.67,242.63,207.70,227.33
3,LAS,"Las Vegas, NV",NV,299459.0,2051.0,350437.0,867.0,85.5,147.90,139.33,144.77,170.65
4,LAX,"Los Angeles, CA",CA,315904.0,2493.0,373393.0,954.0,84.6,205.28,149.98,169.38,206.81
5,ORD,"Chicago, IL",IL,205165.0,1442.0,239114.0,1721.0,85.8,223.18,201.64,217.43,233.31
6,PDX,"Portland, OR",OR,178619.0,2401.0,238323.0,129.0,74.9,178.91,166.86,152.47,191.22
7,PHX,"Phoenix, AZ",AZ,308110.0,2375.0,372050.0,1107.0,82.8,191.28,158.87,164.62,185.09
8,SAN,"San Diego, CA",CA,194524.0,1443.0,233094.0,1050.0,83.5,182.47,171.90,175.79,221.54
9,SFO,"San Francisco, CA",CA,261444.0,2181.0,321273.0,679.0,81.4,154.75,169.95,177.04,192.84


In [7]:
master_top10["DEMAND_RANK"] = (
    master_top10["PASSENGERS"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

master_top10 = master_top10.sort_values(
    "DEMAND_RANK"
)

master_top10

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS,FLIGHTS,SEATS,DISTANCE,LOAD_FACTOR_PCT,FARE_2022,FARE_2023,FARE_2024,FARE_2025,DEMAND_RANK
4,LAX,"Los Angeles, CA",CA,315904.0,2493.0,373393.0,954.0,84.6,205.28,149.98,169.38,206.81,1
7,PHX,"Phoenix, AZ",AZ,308110.0,2375.0,372050.0,1107.0,82.8,191.28,158.87,164.62,185.09,2
3,LAS,"Las Vegas, NV",NV,299459.0,2051.0,350437.0,867.0,85.5,147.90,139.33,144.77,170.65,3
0,ANC,"Anchorage, AK",AK,278351.0,2362.0,331528.0,1448.0,84.0,219.94,205.09,263.05,282.22,4
1,DEN,"Denver, CO",CO,269743.0,2001.0,312259.0,1024.0,86.4,163.52,161.81,163.03,173.53,5
9,SFO,"San Francisco, CA",CA,261444.0,2181.0,321273.0,679.0,81.4,154.75,169.95,177.04,192.84,6
2,DFW,"Dallas/Fort Worth, TX",TX,237870.0,1497.0,268959.0,1660.0,88.4,263.67,242.63,207.70,227.33,7
5,ORD,"Chicago, IL",IL,205165.0,1442.0,239114.0,1721.0,85.8,223.18,201.64,217.43,233.31,8
8,SAN,"San Diego, CA",CA,194524.0,1443.0,233094.0,1050.0,83.5,182.47,171.90,175.79,221.54,9
6,PDX,"Portland, OR",OR,178619.0,2401.0,238323.0,129.0,74.9,178.91,166.86,152.47,191.22,10


In [8]:
yearly_demand = (
    sea_top10
    .groupby(
        ["YEAR", "DEST"],
        as_index=False
    )
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum")
    )
)

yearly_demand["LOAD_FACTOR_PCT"] = (
    yearly_demand["PASSENGERS"]
    / yearly_demand["SEATS"]
    * 100
).round(1)

yearly_demand.head(20)

,YEAR,DEST,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
0,2022,ANC,70370.0,553.0,83164.0,84.6
1,2022,DEN,65105.0,485.0,76245.0,85.4
2,2022,DFW,45928.0,282.0,51872.0,88.5
3,2022,LAS,75279.0,505.0,86550.0,87.0
4,2022,LAX,73103.0,579.0,87102.0,83.9
5,2022,ORD,51337.0,362.0,60990.0,84.2
6,2022,PDX,50963.0,646.0,67423.0,75.6
7,2022,PHX,72822.0,529.0,89954.0,81.0
8,2022,SAN,47280.0,329.0,54249.0,87.2
9,2022,SFO,64780.0,537.0,78150.0,82.9


In [9]:
print("Master destinations:")
print(master_top10["DEST"].nunique())

print("\nMaster shape:")
print(master_top10.shape)

print("\nMissing values:")
print(master_top10.isna().sum())

print("\nYearly demand shape:")
print(yearly_demand.shape)

print("\nYears:")
print(sorted(yearly_demand["YEAR"].unique()))

Master destinations:
10

Master shape:
(10, 13)

Missing values:
DEST               0
DEST_CITY_NAME     0
DEST_STATE_ABR     0
PASSENGERS         0
FLIGHTS            0
SEATS              0
DISTANCE           0
LOAD_FACTOR_PCT    0
FARE_2022          0
FARE_2023          0
FARE_2024          0
FARE_2025          0
DEMAND_RANK        0
dtype: int64

Yearly demand shape:
(40, 6)

Years:
[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [10]:
master_top10.to_csv(
    processed_folder / "master_top10_destinations.csv",
    index=False
)

yearly_demand.to_csv(
    processed_folder / "top10_yearly_demand.csv",
    index=False
)

print("Master datasets saved successfully.")

Master datasets saved successfully.


In [11]:
for file in sorted(processed_folder.glob("*.csv")):
    print(file.name)

airline_by_destination_top10.csv
airline_service_top10.csv
master_top10_destinations.csv
state_demand.csv
top10_2025_balance.csv
top10_airfare_2022_2025.csv
top10_airfare_long_2022_2025.csv
top10_yearly_demand.csv


In [12]:
state_demand = (
    sea_all
    .groupby(
        ["DEST_STATE_ABR", "DEST_STATE_NM"],
        as_index=False
    )
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum")
    )
)

state_demand["LOAD_FACTOR_PCT"] = (
    state_demand["PASSENGERS"]
    / state_demand["SEATS"]
    * 100
).round(1)

state_demand = state_demand.sort_values(
    "PASSENGERS",
    ascending=False
)

state_demand.head(10)

,DEST_STATE_ABR,DEST_STATE_NM,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
2,CA,California,1602543.0,13948.0,1953677.0,82.0
33,TX,Texas,453382.0,3210.0,529504.0,85.6
0,AK,Alaska,447018.0,3949.0,551682.0,81.0
6,HI,Hawaii,397257.0,2325.0,450008.0,88.3
1,AZ,Arizona,345800.0,2703.0,418272.0,82.7
24,NV,Nevada,333560.0,2412.0,392342.0,85.0
28,OR,Oregon,318261.0,4665.0,430510.0,73.9
36,WA,Washington,308016.0,4885.0,390908.0,78.8
4,FL,Florida,284219.0,1760.0,316438.0,89.8
3,CO,Colorado,272170.0,2041.0,315522.0,86.3


In [13]:
airline_service = (
    sea_top10[
        sea_top10["PASSENGERS"] > 0
    ]
    .groupby(
        ["UNIQUE_CARRIER", "UNIQUE_CARRIER_NAME"],
        as_index=False
    )
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum")
    )
)

airline_service["LOAD_FACTOR_PCT"] = (
    airline_service["PASSENGERS"]
    / airline_service["SEATS"]
    * 100
).round(1)

airline_service = airline_service.sort_values(
    "PASSENGERS",
    ascending=False
)

airline_service

,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
2,AS,Alaska Airlines Inc.,1223112.0,8486.0,1450141.0,84.3
3,DL,Delta Air Lines Inc.,503921.0,4128.0,629445.0,80.1
9,UA,United Air Lines Inc.,241239.0,1685.0,276266.0,87.3
1,AA,American Airlines Inc.,197142.0,1241.0,226888.0,86.9
10,WN,Southwest Airlines Co.,137970.0,986.0,162598.0,84.9
7,OO,SkyWest Airlines Inc.,107108.0,1797.0,130387.0,82.1
4,F9,Frontier Airlines Inc.,56696.0,323.0,63984.0,88.6
8,QX,Horizon Air,44208.0,716.0,54416.0,81.2
6,NK,Spirit Air Lines,34999.0,213.0,41039.0,85.3
5,MQ,Envoy Air,2625.0,38.0,2888.0,90.9


In [14]:
airline_by_destination = (
    sea_top10[
        sea_top10["PASSENGERS"] > 0
    ]
    .groupby(
        ["DEST", "UNIQUE_CARRIER", "UNIQUE_CARRIER_NAME"],
        as_index=False
    )
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum")
    )
)

airline_by_destination["LOAD_FACTOR_PCT"] = (
    airline_by_destination["PASSENGERS"]
    / airline_by_destination["SEATS"]
    * 100
).round(1)

airline_by_destination = airline_by_destination.sort_values(
    ["DEST", "PASSENGERS"],
    ascending=[True, False]
)

airline_by_destination.head(20)

,DEST,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
0,ANC,AS,Alaska Airlines Inc.,218368.0,1566.0,258286.0,84.5
1,ANC,DL,Delta Air Lines Inc.,59983.0,481.0,73083.0,82.1
6,DEN,UA,United Air Lines Inc.,83830.0,572.0,93642.0,89.5
2,DEN,AS,Alaska Airlines Inc.,74071.0,491.0,84724.0,87.4
7,DEN,WN,Southwest Airlines Co.,56576.0,402.0,65742.0,86.1
3,DEN,DL,Delta Air Lines Inc.,33419.0,336.0,43678.0,76.5
4,DEN,F9,Frontier Airlines Inc.,19398.0,109.0,21732.0,89.3
5,DEN,OO,SkyWest Airlines Inc.,2449.0,35.0,2552.0,96.0
8,DFW,AA,American Airlines Inc.,144444.0,851.0,161950.0,89.2
9,DFW,AS,Alaska Airlines Inc.,78537.0,513.0,89687.0,87.6


In [15]:
state_demand.to_csv(
    processed_folder / "state_demand.csv",
    index=False
)

airline_service.to_csv(
    processed_folder / "airline_service_top10.csv",
    index=False
)

airline_by_destination.to_csv(
    processed_folder / "airline_by_destination_top10.csv",
    index=False
)

print("Supporting datasets saved successfully.")

Supporting datasets saved successfully.


In [16]:
for file in sorted(processed_folder.glob("*.csv")):
    print(file.name)

airline_by_destination_top10.csv
airline_service_top10.csv
master_top10_destinations.csv
state_demand.csv
top10_2025_balance.csv
top10_airfare_2022_2025.csv
top10_airfare_long_2022_2025.csv
top10_yearly_demand.csv


In [17]:
balance_2025 = (
    yearly_demand[
        yearly_demand["YEAR"] == 2025
    ]
    .copy()
)

balance_2025

,YEAR,DEST,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
30,2025,ANC,62726.0,580.0,75181.0,83.4
31,2025,DEN,65459.0,490.0,74929.0,87.4
32,2025,DFW,69883.0,451.0,79127.0,88.3
33,2025,LAS,66866.0,468.0,78364.0,85.3
34,2025,LAX,75571.0,631.0,93352.0,81.0
35,2025,ORD,52371.0,377.0,63515.0,82.5
36,2025,PDX,37905.0,540.0,49381.0,76.8
37,2025,PHX,77961.0,627.0,94070.0,82.9
38,2025,SAN,45841.0,351.0,54624.0,83.9
39,2025,SFO,65757.0,561.0,82533.0,79.7


In [18]:
balance_2025 = balance_2025.rename(
    columns={
        "PASSENGERS": "PASSENGERS_2025",
        "FLIGHTS": "FLIGHTS_2025",
        "SEATS": "SEATS_2025",
        "LOAD_FACTOR_PCT": "LOAD_FACTOR_2025"
    }
)

balance_2025 = balance_2025.drop(
    columns="YEAR"
)

balance_2025

,DEST,PASSENGERS_2025,FLIGHTS_2025,SEATS_2025,LOAD_FACTOR_2025
30,ANC,62726.0,580.0,75181.0,83.4
31,DEN,65459.0,490.0,74929.0,87.4
32,DFW,69883.0,451.0,79127.0,88.3
33,LAS,66866.0,468.0,78364.0,85.3
34,LAX,75571.0,631.0,93352.0,81.0
35,ORD,52371.0,377.0,63515.0,82.5
36,PDX,37905.0,540.0,49381.0,76.8
37,PHX,77961.0,627.0,94070.0,82.9
38,SAN,45841.0,351.0,54624.0,83.9
39,SFO,65757.0,561.0,82533.0,79.7


In [19]:
balance_2025 = balance_2025.merge(
    airfare[
        ["Dest", "FARE_2025"]
    ],
    left_on="DEST",
    right_on="Dest",
    how="left"
)

balance_2025 = balance_2025.drop(
    columns="Dest"
)

balance_2025

,DEST,PASSENGERS_2025,FLIGHTS_2025,SEATS_2025,LOAD_FACTOR_2025,FARE_2025
0,ANC,62726.0,580.0,75181.0,83.4,282.22
1,DEN,65459.0,490.0,74929.0,87.4,173.53
2,DFW,69883.0,451.0,79127.0,88.3,227.33
3,LAS,66866.0,468.0,78364.0,85.3,170.65
4,LAX,75571.0,631.0,93352.0,81.0,206.81
5,ORD,52371.0,377.0,63515.0,82.5,233.31
6,PDX,37905.0,540.0,49381.0,76.8,191.22
7,PHX,77961.0,627.0,94070.0,82.9,185.09
8,SAN,45841.0,351.0,54624.0,83.9,221.54
9,SFO,65757.0,561.0,82533.0,79.7,192.84


In [20]:
destination_info = master_top10[
    [
        "DEST",
        "DEST_CITY_NAME",
        "DEST_STATE_ABR",
        "DEMAND_RANK"
    ]
]

balance_2025 = balance_2025.merge(
    destination_info,
    on="DEST",
    how="left"
)

balance_2025

,DEST,PASSENGERS_2025,FLIGHTS_2025,SEATS_2025,LOAD_FACTOR_2025,FARE_2025,DEST_CITY_NAME,DEST_STATE_ABR,DEMAND_RANK
0,ANC,62726.0,580.0,75181.0,83.4,282.22,"Anchorage, AK",AK,4
1,DEN,65459.0,490.0,74929.0,87.4,173.53,"Denver, CO",CO,5
2,DFW,69883.0,451.0,79127.0,88.3,227.33,"Dallas/Fort Worth, TX",TX,7
3,LAS,66866.0,468.0,78364.0,85.3,170.65,"Las Vegas, NV",NV,3
4,LAX,75571.0,631.0,93352.0,81.0,206.81,"Los Angeles, CA",CA,1
5,ORD,52371.0,377.0,63515.0,82.5,233.31,"Chicago, IL",IL,8
6,PDX,37905.0,540.0,49381.0,76.8,191.22,"Portland, OR",OR,10
7,PHX,77961.0,627.0,94070.0,82.9,185.09,"Phoenix, AZ",AZ,2
8,SAN,45841.0,351.0,54624.0,83.9,221.54,"San Diego, CA",CA,9
9,SFO,65757.0,561.0,82533.0,79.7,192.84,"San Francisco, CA",CA,6


In [21]:
balance_2025 = balance_2025[
    [
        "DEST",
        "DEST_CITY_NAME",
        "DEST_STATE_ABR",
        "PASSENGERS_2025",
        "FLIGHTS_2025",
        "SEATS_2025",
        "LOAD_FACTOR_2025",
        "FARE_2025",
        "DEMAND_RANK"
    ]
]

balance_2025 = balance_2025.sort_values(
    "PASSENGERS_2025",
    ascending=False
)

balance_2025

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS_2025,FLIGHTS_2025,SEATS_2025,LOAD_FACTOR_2025,FARE_2025,DEMAND_RANK
7,PHX,"Phoenix, AZ",AZ,77961.0,627.0,94070.0,82.9,185.09,2
4,LAX,"Los Angeles, CA",CA,75571.0,631.0,93352.0,81.0,206.81,1
2,DFW,"Dallas/Fort Worth, TX",TX,69883.0,451.0,79127.0,88.3,227.33,7
3,LAS,"Las Vegas, NV",NV,66866.0,468.0,78364.0,85.3,170.65,3
9,SFO,"San Francisco, CA",CA,65757.0,561.0,82533.0,79.7,192.84,6
1,DEN,"Denver, CO",CO,65459.0,490.0,74929.0,87.4,173.53,5
0,ANC,"Anchorage, AK",AK,62726.0,580.0,75181.0,83.4,282.22,4
5,ORD,"Chicago, IL",IL,52371.0,377.0,63515.0,82.5,233.31,8
8,SAN,"San Diego, CA",CA,45841.0,351.0,54624.0,83.9,221.54,9
6,PDX,"Portland, OR",OR,37905.0,540.0,49381.0,76.8,191.22,10


In [22]:
balance_2025 = balance_2025.rename(
    columns={
        "DEMAND_RANK": "OVERALL_DEMAND_RANK"
    }
)

balance_2025["DEMAND_RANK_2025"] = (
    balance_2025["PASSENGERS_2025"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

balance_2025 = balance_2025.sort_values(
    "DEMAND_RANK_2025"
)

balance_2025

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS_2025,FLIGHTS_2025,SEATS_2025,LOAD_FACTOR_2025,FARE_2025,OVERALL_DEMAND_RANK,DEMAND_RANK_2025
7,PHX,"Phoenix, AZ",AZ,77961.0,627.0,94070.0,82.9,185.09,2,1
4,LAX,"Los Angeles, CA",CA,75571.0,631.0,93352.0,81.0,206.81,1,2
2,DFW,"Dallas/Fort Worth, TX",TX,69883.0,451.0,79127.0,88.3,227.33,7,3
3,LAS,"Las Vegas, NV",NV,66866.0,468.0,78364.0,85.3,170.65,3,4
9,SFO,"San Francisco, CA",CA,65757.0,561.0,82533.0,79.7,192.84,6,5
1,DEN,"Denver, CO",CO,65459.0,490.0,74929.0,87.4,173.53,5,6
0,ANC,"Anchorage, AK",AK,62726.0,580.0,75181.0,83.4,282.22,4,7
5,ORD,"Chicago, IL",IL,52371.0,377.0,63515.0,82.5,233.31,8,8
8,SAN,"San Diego, CA",CA,45841.0,351.0,54624.0,83.9,221.54,9,9
6,PDX,"Portland, OR",OR,37905.0,540.0,49381.0,76.8,191.22,10,10


In [23]:
print("Shape:")
print(balance_2025.shape)

print("\nDestinations:")
print(balance_2025["DEST"].tolist())

print("\nMissing values:")
print(balance_2025.isna().sum())

print("\n2025 fare range:")
print(
    balance_2025["FARE_2025"].min(),
    "-",
    balance_2025["FARE_2025"].max()
)

balance_2025[
    [
        "DEST",
        "PASSENGERS_2025",
        "FLIGHTS_2025",
        "FARE_2025",
        "DEMAND_RANK_2025"
    ]
]

Shape:
(10, 10)

Destinations:
['PHX', 'LAX', 'DFW', 'LAS', 'SFO', 'DEN', 'ANC', 'ORD', 'SAN', 'PDX']

Missing values:
DEST                   0
DEST_CITY_NAME         0
DEST_STATE_ABR         0
PASSENGERS_2025        0
FLIGHTS_2025           0
SEATS_2025             0
LOAD_FACTOR_2025       0
FARE_2025              0
OVERALL_DEMAND_RANK    0
DEMAND_RANK_2025       0
dtype: int64

2025 fare range:
170.65 - 282.22


,DEST,PASSENGERS_2025,FLIGHTS_2025,FARE_2025,DEMAND_RANK_2025
7,PHX,77961.0,627.0,185.09,1
4,LAX,75571.0,631.0,206.81,2
2,DFW,69883.0,451.0,227.33,3
3,LAS,66866.0,468.0,170.65,4
9,SFO,65757.0,561.0,192.84,5
1,DEN,65459.0,490.0,173.53,6
0,ANC,62726.0,580.0,282.22,7
5,ORD,52371.0,377.0,233.31,8
8,SAN,45841.0,351.0,221.54,9
6,PDX,37905.0,540.0,191.22,10


In [24]:
balance_2025.to_csv(
    processed_folder / "top10_2025_balance.csv",
    index=False
)

print("2025 balance dataset saved successfully.")

2025 balance dataset saved successfully.


In [25]:
for file in sorted(
    processed_folder.glob("*.csv")
):
    print(file.name)

airline_by_destination_top10.csv
airline_service_top10.csv
master_top10_destinations.csv
state_demand.csv
top10_2025_balance.csv
top10_airfare_2022_2025.csv
top10_airfare_long_2022_2025.csv
top10_yearly_demand.csv
